In [1]:
# Imports and path handling
import os, copy, random, json
from pathlib import Path

import pandas as pd
import numpy as np
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision
from torchvision import transforms, models

# Determine base directory where the competition data lives
BASE_DIR = Path("cassava-leaf-disease-classification")
if not BASE_DIR.exists():
    # fallback to typical Kaggle input location
    BASE_DIR = Path("../input/cassava-leaf-disease-classification")
assert BASE_DIR.exists(), f"Data folder not found at {BASE_DIR}"

TRAIN_CSV = BASE_DIR / "train.csv"
TRAIN_IMG_DIR = BASE_DIR / "train_images"
TEST_IMG_DIR = BASE_DIR / "test_images"
SAMPLE_SUBMIT = BASE_DIR / "sample_submission.csv"




In [2]:
# Simple dataset that reads images based on the train CSV
class CassavaDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.img_dir = Path(img_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.img_dir / row["image_id"]
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # to RGB
        if self.transform:
            img = self.transform(img)
        label = int(row["label"])
        return img, label


# Transformations (same for train/val, only augmentation differs)
train_transform = transforms.Compose(
    [
        transforms.ToPILImage(),
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

val_transform = transforms.Compose(
    [
        transforms.ToPILImage(),
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)



In [3]:
# Load dataset and create train/validation splits
full_dataset = CassavaDataset(TRAIN_CSV, TRAIN_IMG_DIR, transform=train_transform)

val_size = int(0.1 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(
    full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)
# Override transforms for validation part
val_dataset.dataset.transform = val_transform

train_loader = DataLoader(
    train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [4]:
# Model definition (ResNet34 pretrained, final layer adapted to 5 classes)
model = models.resnet34(pretrained=True)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 5)  # 5 disease categories
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)




/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth
100%|██████████| 83.3M/83.3M [00:00<00:00, 115MB/s]


In [5]:
# Simple training loop (few epochs for speed)
def evaluate(loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total if total > 0 else 0.0


epochs = 3
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)

    epoch_loss = running_loss / train_size
    val_acc = evaluate(val_loader)
    print(f"Epoch [{epoch+1}/{epochs}]  Loss: {epoch_loss:.4f}  Val Acc: {val_acc:.4f}")




Epoch [1/3]  Loss: 0.6176  Val Acc: 0.8237
Epoch [2/3]  Loss: 0.3006  Val Acc: 0.7954
Epoch [3/3]  Loss: 0.1334  Val Acc: 0.7954


In [6]:
# Inference on the test set, respecting the order of sample_submission
class TestDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_dir = Path(img_dir)
        self.img_list = sorted(
            [p.name for p in self.img_dir.iterdir() if p.suffix.lower() == ".jpg"]
        )
        self.transform = transform

    def __len__(self):
        return len(self.img_list)

    def __getitem__(self, idx):
        fn = self.img_list[idx]
        img_path = self.img_dir / fn
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(img)
        return img, fn


test_dataset = TestDataset(TEST_IMG_DIR, transform=val_transform)
test_loader = DataLoader(
    test_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True
)

model.eval()
pred_dict = {}
with torch.no_grad():
    for imgs, fns in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        _, preds = torch.max(outputs, 1)
        preds = preds.cpu().numpy()
        for fn, p in zip(fns, preds):
            pred_dict[fn] = int(p)



In [7]:
# Write submission.csv matching the sample format
submission_path = "./submission.csv"
sample_df = pd.read_csv(SAMPLE_SUBMIT)
# Ensure we follow the exact ordering from the sample file
submission_df = pd.DataFrame(
    {
        "image_id": sample_df["image_id"],
        "label": sample_df["image_id"]
        .map(pred_dict)
        .fillna(0)
        .astype(int),  # default to 0 if missing
    }
)
submission_df.to_csv(submission_path, index=False)
print(f"Submission file written to {submission_path} with {len(submission_df)} rows.")



Submission file written to ./submission.csv with 2676 rows.


In [8]:
# Quick sanity check: read back the file
print(pd.read_csv(submission_path).head())

         image_id  label
0  1234294272.jpg      3
1  1234332763.jpg      3
2  1234375577.jpg      1
3  1234555380.jpg      1
4  1234571117.jpg      3
